<a href="https://colab.research.google.com/github/shiitavie/face/blob/master/notebooks/stage1a_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

REPO = "https://github.com/shiitavie/face.git"
BRANCH = "master"

import os
import subprocess

if os.path.exists("/content/face"):
    os.chdir("/content/face")
    result = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True)
else:
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO, "/content/face"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        os.chdir("/content/face")

if result.returncode != 0:
    raise SystemExit(f"git failed (exit {result.returncode}):\n{result.stderr}")

!git log --oneline -1

## 1. Check the GPU

In [5]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n{name}  |  {vram:.1f} GB")

if vram < 20:
    raise RuntimeError(
        f"{name} has only {vram:.1f} GB. A 7B VLM at fp16 needs ~17 GB of weights "
        "alone. Switch to an L4 (24 GB) or A100. Do not work around this by "
        "quantizing -- see the note at the top."
    )

name, memory.total [MiB]
NVIDIA L4, 23034 MiB

NVIDIA L4  |  23.7 GB


## 2. Get the code

The repo is public, so no authentication is needed. Re-running this cell pulls
the latest commits rather than re-cloning.

In [32]:
REPO = "https://github.com/shiitavie/face.git"
BRANCH = "master"

import os
import subprocess

if os.path.exists("/content/face"):
    os.chdir("/content/face")
    result = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True)
else:
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO, "/content/face"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        os.chdir("/content/face")

if result.returncode != 0:
    raise SystemExit(f"git failed (exit {result.returncode}):\n{result.stderr}")

!git log --oneline -1

daeb1cd (HEAD -> master, origin/master, origin/HEAD) Report measurement reliability and correct for attenuation


## 3. Install dependencies

Editable install so `facecav` imports from the scripts. Torch is intentionally
not reinstalled — Colab's build is CUDA-matched and replacing it breaks CUDA.

In [7]:
!pip install -q -e ".[dev]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 77.6 MB/s eta 0:00:00
  Building editable for facecav (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1

## 4. Point at the CFD images in your Drive

Set `DRIVE_CFD` to your unzipped `CFD Version 3.0` folder. No zipping needed.

**Reading straight from Drive is the default and is usually the right choice.**
A forward pass costs ~0.5–2 s per image, so Drive's per-file latency adds only a
few minutes across a full run — less than copying ~2 GB would cost. And Colab
wipes local storage between sessions, so a copy is paid again every time.

Set `COPY_LOCAL = True` only if you hit repeated `Input/output error`s from
Drive. Even then it is rarely urgent: the driver resumes from its JSONL, so a
failed read costs one image and a rerun. When enabled, it copies only the
workbook and the 831 neutral images, not the ~610 expression variants and the
7 MB measurement PDF that the pipeline never touches.

In [9]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CFD = "/content/drive/MyDrive/CFD Version 3.0"   # <-- adjust to yours
COPY_LOCAL = False

import shutil
from pathlib import Path

WORKBOOK = "CFD 3.0 Norming Data and Codebook.xlsx"
drive_root = Path(DRIVE_CFD)

if not (drive_root / WORKBOOK).exists():
    print(f"Not found: {drive_root / WORKBOOK}\nSearching Drive...")
    hits = list(Path("/content/drive/MyDrive").glob(f"**/{WORKBOOK}"))
    raise SystemExit(
        "Set DRIVE_CFD to one of:\n  " + "\n  ".join(str(h.parent) for h in hits)
        if hits
        else "Could not find the CFD workbook anywhere in MyDrive."
    )

if COPY_LOCAL:
    CFD_ROOT = Path("/content/cfd/CFD Version 3.0")
    if not (CFD_ROOT / WORKBOOK).exists():
        CFD_ROOT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_root / WORKBOOK, CFD_ROOT / WORKBOOK)
        for src in drive_root.rglob("*-N.jpg"):
            dst = CFD_ROOT / src.relative_to(drive_root)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
else:
    CFD_ROOT = drive_root

CFD_ROOT_STR = str(CFD_ROOT)
print(f"{CFD_ROOT}\n{sum(1 for _ in CFD_ROOT.rglob('*-N.jpg'))} neutral images (expect 831)")

Mounted at /content/drive
/content/drive/MyDrive/CFD Version 3.0
831 neutral images (expect 831)


## 5. Verify the pipeline before spending GPU time

Runs the test suite and builds the manifest — no GPU, no model download.
Expect **25 passing** and **831 rows / 826 matched**. If this fails, stop:
nothing downstream can be right.

In [10]:
!python -m pytest tests/ -q

from facecav.data.cfd import build_manifest

manifest = build_manifest(CFD_ROOT)
print(f"\nrows: {len(manifest)}   matched: {(manifest.join_status == 'matched').sum()}")
print(manifest.join_status.value_counts().to_string())

....ssssss...............                                                [100%]
19 passed, 6 skipped in 2.14s

rows: 831   matched: 826
join_status
matched           826
no_norming_row      5


## 6. Smoke test — three images

The first real forward pass, and where the untested assumptions in `rater.py`
surface: whether the chat template renders, whether the processor accepts the
image, and whether the logits land on the position after `"The rating is "`.

**What to check, worst first:**
- **Identical ratings across all three faces** → the image is not reaching the
  model, and every downstream number would be an artifact of the prompt alone.
- **`refusal_mass` near 1.0** → the model is not answering with a digit.
- **Rating outside [1, 7]** → impossible by construction; token IDs resolved wrong.

These are the first three manifest rows (all Asian female), so read the wiring,
not the values.

In [11]:
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

!python experiments/stage1a_rate_cfd.py --model "$MODEL" --cfd-root "$CFD_ROOT_STR" --limit 3

import json
from pathlib import Path

out = Path("artifacts/stage1a") / f"{MODEL.replace('/', '__')}.jsonl"
for line in out.read_text().splitlines():
    r = json.loads(line)
    print(f"{r['model_id']:>12}  rating={r['expected_rating']:.3f}  "
          f"refusal={r['refusal_mass']:.4f}  probs={[round(p, 3) for p in r['rating_probs']]}")

2026-09-19 14:26:07.346431: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-19 14:26:07.418286: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
preprocessor_config.json: 100% 350/350 [00:00<00:00, 3.06MB/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False

In [ ]:
!python experiments/reliability_battery.py --model "$MODEL" --cfd-root "$CFD_ROOT_STR" \
    --n-images 40 --n-samples 32 --temperature 0.3 --compare-logits


2026-09-19 23:52:57.577919: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-19 23:52:57.649569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a 

## 7. Full Stage 1a

All 831 images. Resumable: results append to JSONL and completed images are
skipped, so a disconnect costs at most one image — just rerun this cell.

In [ ]:
!python experiments/stage1a_rate_cfd.py --model "$MODEL" --cfd-root "$CFD_ROOT_STR"

## 8. First look

In [ ]:
import pandas as pd

ratings = pd.read_json(out, lines=True)
print(f"n = {len(ratings)}")
print(f"\nrefusal mass: mean={ratings.refusal_mass.mean():.4f}  max={ratings.refusal_mass.max():.4f}")

print("\nmean expected rating by race x gender:")
print(pd.crosstab(ratings.race_code, ratings.gender_code,
                  values=ratings.expected_rating, aggfunc="mean").round(3).to_string())

print("\nrefusal by race (differential refusal is itself a finding -- spec 5.6):")
print(ratings.groupby("race_code").refusal_mass.mean().round(4).to_string())

## 9. Save results back to Drive

Colab storage is ephemeral. The JSONL is small — always copy it out.

In [ ]:
!mkdir -p "/content/drive/MyDrive/face_artifacts"
!cp -r artifacts/stage1a "/content/drive/MyDrive/face_artifacts/"
!ls -la "/content/drive/MyDrive/face_artifacts/stage1a"

---
### Next

Repeat cell 7 for `OpenGVLab/InternVL3-8B` and
`HuggingFaceM4/Idefics3-8B-Llama3`. The 32B/38B arm needs an A100 80GB
(High-RAM toggle).

**Cell 8 is not a result yet.** Spec §9 requires prompt-paraphrase stability and
refusal rates first — a mean-rating table looks publishable while resting on a
single untested prompt. The ICL condition does not exist yet; wiring it needs
the §14 decision on demonstration set size and composition.